<a href="https://colab.research.google.com/github/josemanuelmontanoortiz978-droid/PROYECTOS-ED1/blob/main/Ejercicios_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ADT POLINOMIO

In [ ]:
from __future__ import annotations
from typing import Optional


class NodoTermino:
    """Representa un monomio (coef * x^exp) dentro de una lista enlazada."""

    def __init__(self, coef: float, exp: int, siguiente: Optional[NodoTermino] = None) -> None:
        self.coef: float = coef
        self.exp: int = exp
        self.siguiente: Optional[NodoTermino] = siguiente


class Polinomio:
    """ADT Polinomio implementado mediante lista enlazada ordenada por exponente descendente."""

    def __init__(self) -> None:
        """Inicializa un polinomio nulo P(x) = 0."""
        self._cabeza: Optional[NodoTermino] = None

    def agregar_termino(self, coef: float, exp: int) -> None:
        """Inserta o acumula un término manteniendo orden descendente y eliminando ceros."""
        if exp < 0:
            raise ValueError("El exponente de un polinomio debe ser un entero no negativo.")
        if abs(coef) < 1e-12:
            return

        # Caso 1: Lista vacía o el nuevo exponente es mayor que la cabeza actual
        if self._cabeza is None or exp > self._cabeza.exp:
            self._cabeza = NodoTermino(coef, exp, self._cabeza)
            return

        # Caso 2: Coincide con el exponente de la cabeza
        if self._cabeza.exp == exp:
            self._cabeza.coef += coef
            if abs(self._cabeza.coef) < 1e-12:
                self._cabeza = self._cabeza.siguiente
            return

        # Caso 3: Buscar posición intermedia o final
        anterior: NodoTermino = self._cabeza
        actual: Optional[NodoTermino] = self._cabeza.siguiente

        while actual is not None and actual.exp > exp:
            anterior = actual
            actual = actual.siguiente

        if actual is not None and actual.exp == exp:
            actual.coef += coef
            if abs(actual.coef) < 1e-12:
                anterior.siguiente = actual.siguiente
        else:
            anterior.siguiente = NodoTermino(coef, exp, actual)

    def evaluar(self, x: float) -> float:
        """Evalúa P(x) recorriendo únicamente los términos no nulos."""
        resultado: float = 0.0
        actual: Optional[NodoTermino] = self._cabeza
        while actual is not None:
            resultado += actual.coef * (x ** actual.exp)
            actual = actual.siguiente
        return resultado

    def sumar(self, otro: Polinomio) -> Polinomio:
        """Retorna un nuevo Polinomio resultante de P(x) + Q(x) en tiempo O(n + m)."""
        resultado: Polinomio = Polinomio()
        p1: Optional[NodoTermino] = self._cabeza
        p2: Optional[NodoTermino] = otro._cabeza

        # Puntero auxiliar de cola para insertar en O(1) aprovechando el orden
        cola_res: Optional[NodoTermino] = None

        def _anexar_ordenado(c: float, e: int) -> None:
            nonlocal cola_res
            if abs(c) < 1e-12:
                return
            nuevo = NodoTermino(c, e)
            if resultado._cabeza is None:
                resultado._cabeza = nuevo
                cola_res = nuevo
            else:
                assert cola_res is not None
                cola_res.siguiente = nuevo
                cola_res = nuevo

        while p1 is not None and p2 is not None:
            if p1.exp > p2.exp:
                _anexar_ordenado(p1.coef, p1.exp)
                p1 = p1.siguiente
            elif p2.exp > p1.exp:
                _anexar_ordenado(p2.coef, p2.exp)
                p2 = p2.siguiente
            else:
                _anexar_ordenado(p1.coef + p2.coef, p1.exp)
                p1 = p1.siguiente
                p2 = p2.siguiente

        while p1 is not None:
            _anexar_ordenado(p1.coef, p1.exp)
            p1 = p1.siguiente

        while p2 is not None:
            _anexar_ordenado(p2.coef, p2.exp)
            p2 = p2.siguiente

        return resultado

    def restar(self, otro: Polinomio) -> Polinomio:
        """Retorna un nuevo Polinomio resultante de P(x) - Q(x)."""
        negativo_otro: Polinomio = Polinomio()
        actual: Optional[NodoTermino] = otro._cabeza
        while actual is not None:
            negativo_otro.agregar_termino(-actual.coef, actual.exp)
            actual = actual.siguiente
        return self.sumar(negativo_otro)

    def derivar(self) -> Polinomio:
        """Retorna la primera derivada P'(x) aplicando la regla de la potencia."""
        derivada: Polinomio = Polinomio()
        actual: Optional[NodoTermino] = self._cabeza
        while actual is not None:
            if actual.exp > 0:
                derivada.agregar_termino(actual.coef * actual.exp, actual.exp - 1)
            actual = actual.siguiente
        return derivada

    def __str__(self) -> str:
        """Representación algebraica legible en texto."""
        if self._cabeza is None:
            return "0"

        partes: list[str] = []
        actual: Optional[NodoTermino] = self._cabeza
        while actual is not None:
            c: float = actual.coef
            e: int = actual.exp
            signo: str = "-" if c < 0 else "+"
            magnitud: float = abs(c)
            valor_str: str = f"{int(magnitud)}" if magnitud.is_integer() else f"{magnitud:.2f}"

            if e == 0:
                termino = f"{valor_str}"
            elif e == 1:
                termino = "x" if valor_str == "1" else f"{valor_str}x"
            else:
                termino = f"x^{e}" if valor_str == "1" else f"{valor_str}x^{e}"

            if not partes:
                partes.append(f"-{termino}" if c < 0 else termino)
            else:
                partes.append(f" {signo} {termino}")

            actual = actual.siguiente

        return "".join(partes)


# --- PRUEBAS CON 3 POLINOMIOS DISTINTOS ---
p1 = Polinomio()
p1.agregar_termino(3, 4)
p1.agregar_termino(-2, 2)
p1.agregar_termino(5, 0)

p2 = Polinomio()
p2.agregar_termino(2, 2)   # Se cancelará con -2x^2 de P1 al sumar
p2.agregar_termino(4, 3)
p2.agregar_termino(-1, 1)

p3 = Polinomio()
p3.agregar_termino(1.5, 5)
p3.agregar_termino(-7, 0)

print(f"P1(x) = {p1}")
print(f"P2(x) = {p2}")
print(f"P3(x) = {p3}")
print(f"P1(x) + P2(x) = {p1.sumar(p2)}")
print(f"P1(x) - P3(x) = {p1.restar(p3)}")
print(f"P1'(x)        = {p1.derivar()}")
print(f"P1(2)         = {p1.evaluar(2)}")

ADT CONJUNTO

In [ ]:
from typing import Any, Iterable, Iterator


class NodoConjunto:
    """Nodo individual para almacenar un elemento único del conjunto."""

    def __init__(self, dato: Any, siguiente: Optional[NodoConjunto] = None) -> None:
        self.dato: Any = dato
        self.siguiente: Optional[NodoConjunto] = siguiente


class ConjuntoADT:
    """Implementación desde cero del Tipo Abstracto de Datos Conjunto."""

    def __init__(self, elementos: Optional[Iterable[Any]] = None) -> None:
        self._cabeza: Optional[NodoConjunto] = None
        self._tamanio: int = 0
        if elementos is not None:
            for item in elementos:
                self.agregar(item)

    def contiene(self, elemento: Any) -> bool:
        """Verifica pertenencia (elemento ∈ A) en tiempo O(n)."""
        actual: Optional[NodoConjunto] = self._cabeza
        while actual is not None:
            if actual.dato == elemento:
                return True
            actual = actual.siguiente
        return False

    def agregar(self, elemento: Any) -> None:
        """Inserta el elemento solo si no existe previamente en el conjunto."""
        if not self.contiene(elemento):
            self._cabeza = NodoConjunto(elemento, self._cabeza)
            self._tamanio += 1

    def union(self, otro: ConjuntoADT) -> ConjuntoADT:
        """Retorna A ∪ B."""
        resultado: ConjuntoADT = ConjuntoADT()
        for item in self:
            resultado.agregar(item)
        for item in otro:
            resultado.agregar(item)
        return resultado

    def interseccion(self, otro: ConjuntoADT) -> ConjuntoADT:
        """Retorna A ∩ B."""
        resultado: ConjuntoADT = ConjuntoADT()
        for item in self:
            if otro.contiene(item):
                resultado.agregar(item)
        return resultado

    def diferencia(self, otro: ConjuntoADT) -> ConjuntoADT:
        """Retorna A - B (elementos en A que no están en B)."""
        resultado: ConjuntoADT = ConjuntoADT()
        for item in self:
            if not otro.contiene(item):
                resultado.agregar(item)
        return resultado

    def diferencia_simetrica(self, otro: ConjuntoADT) -> ConjuntoADT:
        """Retorna A Δ B = (A - B) ∪ (B - A)."""
        return self.diferencia(otro).union(otro.diferencia(self))

    def es_subconjunto(self, otro: ConjuntoADT) -> bool:
        """Retorna True si A ⊆ B (todo elemento de self está en otro)."""
        if self._tamanio > otro._tamanio:
            return False
        for item in self:
            if not otro.contiene(item):
                return False
        return True

    def __iter__(self) -> Iterator[Any]:
        actual: Optional[NodoConjunto] = self._cabeza
        while actual is not None:
            yield actual.dato
            actual = actual.siguiente

    def __len__(self) -> int:
        return self._tamanio

    def __str__(self) -> str:
        elementos: list[str] = [repr(item) for item in self]
        return "{" + ", ".join(elementos) + "}"


# --- CASOS DE PRUEBA (2 NUMÉRICOS + 1 CON CADENAS) ---
# Caso 1: Enteros con solapamiento parcial
num_a = ConjuntoADT([1, 2, 3, 4, 4, 2])  # Duplicados ignorados automáticamente
num_b = ConjuntoADT([3, 4, 5, 6])
print("=== Caso Numérico 1 ===")
print(f"A = {num_a} | B = {num_b}")
print(f"Unión (A ∪ B):                {num_a.union(num_b)}")
print(f"Intersección (A ∩ B):         {num_a.interseccion(num_b)}")
print(f"Diferencia (A - B):           {num_a.diferencia(num_b)}")
print(f"Diferencia Simétrica (A Δ B): {num_a.diferencia_simetrica(num_b)}")
print(f"¿A ⊆ B?:                      {num_a.es_subconjunto(num_b)}")

# Caso 2: Subconjunto numérico estricto
pares_pequenos = ConjuntoADT([2, 4])
pares_mayores = ConjuntoADT([2, 4, 6, 8, 10])
print("\n=== Caso Numérico 2 (Subconjunto) ===")
print(f"¿{pares_pequenos} ⊆ {pares_mayores}?: {pares_pequenos.es_subconjunto(pares_mayores)}")

# Caso 3: Cadenas (Etiquetas de habilidades)
dev_backend = ConjuntoADT(["Python", "SQL", "Docker", "Git"])
dev_datos = ConjuntoADT(["Python", "SQL", "Spark", "Airflow"])
print("\n=== Caso Cadenas (Perfiles Técnicos) ===")
print(f"Comunes (∩):        {dev_backend.interseccion(dev_datos)}")
print(f"Exclusivas (Δ):     {dev_backend.diferencia_simetrica(dev_datos)}")

MATRIZ DISPERSA

In [ ]:
import sys
import time
import random


class MatrizDispersa:
    """Matriz dispersa respaldada por un diccionario de coordenadas (i, j) -> valor."""

    def __init__(self, filas: int, columnas: int) -> None:
        if filas <= 0 or columnas <= 0:
            raise ValueError("Las dimensiones de la matriz deben ser mayores a 0.")
        self.filas: int = filas
        self.columnas: int = columnas
        self._datos: dict[tuple[int, int], float] = {}

    def _validar_indices(self, i: int, j: int) -> None:
        if not (0 <= i < self.filas and 0 <= j < self.columnas):
            raise IndexError(f"Coordenada ({i}, {j}) fuera de rango ({self.filas}x{self.columnas}).")

    def establecer(self, i: int, j: int, valor: float) -> None:
        """Asigna un valor en (i, j) en tiempo O(1) promedio. Si es 0, libera la clave."""
        self._validar_indices(i, j)
        if abs(valor) < 1e-12:
            self._datos.pop((i, j), None)
        else:
            self._datos[(i, j)] = valor

    def obtener(self, i: int, j: int) -> float:
        """Retorna el valor en (i, j) en O(1) promedio; 0.0 si no está almacenado."""
        self._validar_indices(i, j)
        return self._datos.get((i, j), 0.0)

    def transponer(self) -> MatrizDispersa:
        """Retorna la matriz transpuesta A^T en tiempo O(nnz), donde nnz = no-ceros."""
        transpuesta: MatrizDispersa = MatrizDispersa(self.columnas, self.filas)
        for (i, j), valor in self._datos.items():
            transpuesta._datos[(j, i)] = valor
        return transpuesta

    def densidad(self) -> float:
        """Calcula la proporción de elementos no nulos respecto al total de celdas."""
        total_celdas: int = self.filas * self.columnas
        return len(self._datos) / total_celdas

    @property
    def no_ceros(self) -> int:
        return len(self._datos)


# --- EVALUACIÓN DE EFICIENCIA: MATRIZ DISPERSA VS. MATRIZ DENSA (500 x 500) ---
N_FILAS, N_COLS = 500, 500
ELEMENTOS_NO_CERO = 2_500  # 1% de densidad (250,000 celdas totales)

dispersa = MatrizDispersa(N_FILAS, N_COLS)
densa: list[list[float]] = [[0.0 for _ in range(N_COLS)] for _ in range(N_FILAS)]

random.seed(10)
for _ in range(ELEMENTOS_NO_CERO):
    r = random.randint(0, N_FILAS - 1)
    c = random.randint(0, N_COLS - 1)
    val = round(random.uniform(1.0, 100.0), 2)
    dispersa.establecer(r, c, val)
    densa[r][c] = val

# Medición de memoria aproximada
mem_dispersa = sys.getsizeof(dispersa) + sys.getsizeof(dispersa._datos) + sum(
    sys.getsizeof(k) + sys.getsizeof(v) for k, v in dispersa._datos.items()
)
mem_densa = sys.getsizeof(densa) + sum(
    sys.getsizeof(fila) + sum(sys.getsizeof(celda) for celda in fila) for fila in densa
)

# Medición de tiempo de transposición
t0 = time.perf_counter()
_ = dispersa.transponer()
t_transp_dispersa = (time.perf_counter() - t0) * 1_000  # ms

t0 = time.perf_counter()
_ = [[densa[i][j] for i in range(N_FILAS)] for j in range(N_COLS)]
t_transp_densa = (time.perf_counter() - t0) * 1_000  # ms

print(f"Dimensiones: {N_FILAS}x{N_COLS} ({N_FILAS * N_COLS:,} celdas) | Densidad: {dispersa.densidad():.2%}")
print(f"Memoria MatrizDispersa : {mem_dispersa / 1024:>8.2f} KB")
print(f"Memoria Matriz Densa   : {mem_densa / 1024:>8.2f} KB (Ahorro: {(1 - mem_dispersa / mem_densa):.1%})")
print(f"Transponer Dispersa    : {t_transp_dispersa:>8.4f} ms  [Complejidad O(nnz)]")
print(f"Transponer Densa       : {t_transp_densa:>8.4f} ms  [Complejidad O(filas * cols)]")

INTEGRACION

In [ ]:
class SistemaInventarioRegional:
    """Integra ConjuntoADT, MatrizDispersa y Polinomio para gestión de inventario."""

    def __init__(self, sucursales: list[str], productos: list[str]) -> None:
        self.sucursales: list[str] = sucursales
        self.productos: list[str] = productos
        self.stock: MatrizDispersa = MatrizDispersa(len(sucursales), len(productos))
        self.categorias_por_sucursal: dict[str, ConjuntoADT] = {
            suc: ConjuntoADT() for suc in sucursales
        }
        self.curvas_demanda: dict[str, Polinomio] = {}

    def registrar_stock(self, sucursal: str, producto: str, categoria: str, cantidad: int) -> None:
        """Actualiza el inventario disperso y asocia la categoría a la sucursal."""
        idx_suc: int = self.sucursales.index(sucursal)
        idx_prod: int = self.productos.index(producto)
        self.stock.establecer(idx_suc, idx_prod, float(cantidad))
        if cantidad > 0:
            self.categorias_por_sucursal[sucursal].agregar(categoria)

    def configurar_modelo_demanda(self, producto: str, polinomio: Polinomio) -> None:
        """Asigna un polinomio de proyección de demanda D(t) a un producto."""
        self.curvas_demanda[producto] = polinomio

    def auditar_abastecimiento(self, sucursal_a: str, sucursal_b: str, mes_t: int) -> None:
        """Cruza categorías con ConjuntoADT y evalúa riesgo de quiebre con Polinomio."""
        cat_a: ConjuntoADT = self.categorias_por_sucursal[sucursal_a]
        cat_b: ConjuntoADT = self.categorias_por_sucursal[sucursal_b]

        print(f"=== AUDITORÍA REGIONAL (Mes proyección t = {mes_t}) ===")
        print(f"Catálogo global activo (A ∪ B):      {cat_a.union(cat_b)}")
        print(f"Categorías exclusivas de {sucursal_a}: {cat_a.diferencia(cat_b)}")
        print(f"Densidad actual matriz de stock:     {self.stock.densidad():.2%} ({self.stock.no_ceros} registros activos)\n")

        idx_a: int = self.sucursales.index(sucursal_a)
        for j, prod in enumerate(self.productos):
            stock_actual: float = self.stock.obtener(idx_a, j)
            if prod in self.curvas_demanda:
                demanda_estimada: float = self.curvas_demanda[prod].evaluar(mes_t)
                estado: str = "OK" if stock_actual >= demanda_estimada else f"REABASTECER (+{demanda_estimada - stock_actual:.0f} uds)"
                print(f" - [{sucursal_a}] {prod:<15} | Stock: {stock_actual:>5.0f} | Demanda D({mes_t}): {demanda_estimada:>6.1f} -> {estado}")


# --- EJECUCIÓN DEL CASO APLICADO ---
sucursales_lista = ["Norte", "Sur", "Centro", "Equipetrol"]
productos_lista = ["Laptop Pro", "Monitor 4K", "Teclado Mecánico", "Servidor Rack", "Router WiFi 7", "Switch PoE"]

sistema = SistemaInventarioRegional(sucursales_lista, productos_lista)

# Cargamos solo existencias reales (el resto permanece implícito en 0 en MatrizDispersa)
sistema.registrar_stock("Norte", "Laptop Pro", "Computación", 45)
sistema.registrar_stock("Norte", "Router WiFi 7", "Redes", 18)
sistema.registrar_stock("Sur", "Laptop Pro", "Computación", 30)
sistema.registrar_stock("Sur", "Servidor Rack", "Infraestructura", 5)

# Definimos curva polinómica de demanda para Laptop Pro: D(t) = 2t^2 + 5t + 15
demanda_laptop = Polinomio()
demanda_laptop.agregar_termino(2, 2)
demanda_laptop.agregar_termino(5, 1)
demanda_laptop.agregar_termino(15, 0)
sistema.configurar_modelo_demanda("Laptop Pro", demanda_laptop)

# Definimos curva lineal de demanda para Router WiFi 7: D(t) = 6t + 4
demanda_router = Polinomio()
demanda_router.agregar_termino(6, 1)
demanda_router.agregar_termino(4, 0)
sistema.configurar_modelo_demanda("Router WiFi 7", demanda_router)

# Ejecutamos análisis para el mes t = 3
sistema.auditar_abastecimiento("Norte", "Sur", mes_t=3)